# Module 06 — Convolutional Neural Networks
## 06-07: Semantic & Instance Segmentation

**Objective:** Implement RoI Pooling and RoI Align from scratch, build a mask head, and assemble a toy Mask R-CNN-style pipeline that jointly detects and segments object instances.

**Prerequisites:** 09-01 (IoU, NMS, anchor generation), 09-02 (single-stage detection, anchor assignment), 09-03 (semantic segmentation, FCN/U-Net, pixel-wise masks)

## Part 0 — Setup & Prerequisites

Instance segmentation sits at the intersection of object detection and semantic segmentation. Where 09-03 assigned a class label to every pixel without distinguishing between objects of the same class, **instance segmentation** detects each object individually and predicts a separate binary mask for it. This notebook implements the two critical building blocks — **RoI Pooling** and **RoI Align** — from scratch, then wraps them into a complete toy Mask R-CNN–style pipeline trained on synthetic data.

We first survey the three dense-prediction task families (semantic, instance, panoptic), then implement the components in order of increasing complexity.

**Prerequisites:** 09-01 (IoU, NMS), 09-02 (SSD-style detection), 09-03 (U-Net, pixel-wise CE loss)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import warnings
import random
import time
from typing import Optional

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility & Device ──────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
IMAGE_SIZE       = 128          # Input image H = W
FEATURE_SIZE     = 16           # Backbone output spatial dim (128 // 8)
IN_CHANNELS      = 3            # RGB
BACKBONE_CHANNELS = 64          # Feature map channels from backbone
NUM_CLASSES      = 3            # Foreground classes (+ background = 4 total)
ROI_OUTPUT_SIZE  = 7            # RoI Pool / Align output H = W
MASK_SIZE        = 14           # Mask head output H = W
NUM_TRAIN        = 600          # Training samples
NUM_VAL          = 150          # Validation samples
NUM_TEST         = 150          # Test samples
BATCH_SIZE       = 4            # Small due to variable-length target lists
NUM_EPOCHS       = 6
LEARNING_RATE    = 1e-3
MAX_INSTANCES    = 4            # Max instances per image
MIN_BOX_SIZE     = 16           # Minimum GT box side length (pixels)
HIDDEN_DIM       = 64           # Mask head hidden channels
FC_DIM           = 256          # Classification / regression head FC size
SAMPLING_RATIO   = 2            # RoI Align sampling points per bin

### Task Taxonomy — Semantic vs Instance vs Panoptic Segmentation

Dense-prediction tasks all operate at the pixel level, but they differ in **what label each pixel receives**:

| Task | Per-pixel label | Instance distinct? | Ownership |
|------|-----------------|--------------------|----------|
| **Semantic segmentation** | Class (e.g., *car*, *road*) | No — all cars share one label | 09-03 |
| **Instance segmentation** | Binary mask per instance | Yes — car #1 ≠ car #2 | **09-04 (this notebook)** |
| **Panoptic segmentation** | Class + instance ID | Yes for *things*; class-only for *stuff* | Extension of 09-04 |

**Stuff** classes (sky, road, grass) are amorphous and uncountable — semantic labelling suffices.  
**Things** classes (person, car, dog) are countable and individually identifiable — instance masks are needed.

Panoptic segmentation unifies both: each pixel gets a `(class_id, instance_id)` pair where `instance_id = 0` for stuff classes.

#### Visual comparison (ASCII art)

```
Input image           Semantic seg          Instance seg          Panoptic seg
┌──────────────┐      ┌──────────────┐      ┌──────────────┐      ┌──────────────┐
│  [A]   [A]   │      │  [cls1][cls1]│      │  [A1]  [A2]  │      │  [cls1,1]    │
│              │  →   │              │  →   │              │  →   │  [cls1,2]    │
│     [B]      │      │     [cls2]   │      │     [B1]     │      │  [cls2,1]    │
└──────────────┘      └──────────────┘      └──────────────┘      └──────────────┘
Two objects A,A       Both A pixels get      A pixels split        A pixels split,
and one B.            the same label.        into A1, A2.          B gets (cls2,1).
```

The dominant instance segmentation architecture is **Mask R-CNN** (He et al., 2017), which extends Faster R-CNN with a parallel mask prediction branch. The two key innovations are:
1. **RoI Align** — fixes the quantisation error in RoI Pooling.
2. **Per-class mask head** — predicts one binary mask per class per proposal.

### Synthetic Dataset

We generate images of size $3 \times 128 \times 128$ containing 1–4 non-overlapping coloured rectangles (instances). Each instance has:
- A **bounding box** $(x_1, y_1, x_2, y_2)$ in image pixel coordinates.
- A **binary mask** of shape $(128, 128)$ with 1s inside the box.
- A **class label** drawn from $\{0, 1, 2\}$ (NUM_CLASSES = 3).

Ground truth is stored as a list of dicts: `[{'box': Tensor(4,), 'class': int, 'mask': Tensor(H, W)}, ...]`.

In [ ]:
def generate_non_overlapping_boxes(
    num_instances: int,
    image_size: int,
    min_size: int,
    rng: np.random.Generator,
) -> list[tuple[int, int, int, int]]:
    """Generate non-overlapping bounding boxes within an image.

    Uses rejection sampling: propose a random box and accept only if it does
    not overlap with any previously accepted box.

    Args:
        num_instances: Number of boxes to generate.
        image_size: Height and width of the square image.
        min_size: Minimum side length of each box.
        rng: NumPy random generator for reproducibility.

    Returns:
        List of (x1, y1, x2, y2) tuples in pixel coordinates.
    """
    boxes: list[tuple[int, int, int, int]] = []
    max_attempts = 200

    for _ in range(num_instances):
        for _attempt in range(max_attempts):
            max_start = image_size - min_size - 1
            x1 = int(rng.integers(0, max_start))
            y1 = int(rng.integers(0, max_start))
            max_w = min(image_size - x1, 48)
            max_h = min(image_size - y1, 48)
            if max_w < min_size or max_h < min_size:
                continue
            w = int(rng.integers(min_size, max_w + 1))
            h = int(rng.integers(min_size, max_h + 1))
            x2 = x1 + w
            y2 = y1 + h
            # Check overlap with existing boxes
            overlaps = False
            for bx1, by1, bx2, by2 in boxes:
                inter_x1 = max(x1, bx1)
                inter_y1 = max(y1, by1)
                inter_x2 = min(x2, bx2)
                inter_y2 = min(y2, by2)
                if inter_x1 < inter_x2 and inter_y1 < inter_y2:
                    overlaps = True
                    break
            if not overlaps:
                boxes.append((x1, y1, x2, y2))
                break
    return boxes


def make_instance_segmentation_sample(
    image_size: int,
    num_classes: int,
    max_instances: int,
    min_box_size: int,
    rng: np.random.Generator,
) -> tuple[torch.Tensor, list[dict]]:
    """Generate one synthetic instance segmentation sample.

    Each instance is a filled rectangle with a distinct colour drawn from a
    class-specific palette. The background is a low-intensity noise texture.

    Args:
        image_size: Height and width of the output image in pixels.
        num_classes: Number of foreground object classes.
        max_instances: Maximum number of instances to place.
        min_box_size: Minimum bounding-box side length in pixels.
        rng: NumPy random generator.

    Returns:
        Tuple of:
          - image: Float tensor of shape (3, H, W) in [0, 1].
          - targets: List of dicts, each with keys
              'box'   -> Tensor(4,) [x1, y1, x2, y2] float,
              'class' -> int class index in [0, num_classes),
              'mask'  -> Tensor(H, W) binary float.
    """
    # Soft background noise
    image_np = rng.uniform(0.0, 0.15, (3, image_size, image_size)).astype(np.float32)

    # Class-specific base colours (R, G, B) for foreground instances
    class_colours = [
        np.array([0.85, 0.15, 0.15], dtype=np.float32),  # class 0 — red
        np.array([0.15, 0.75, 0.15], dtype=np.float32),  # class 1 — green
        np.array([0.15, 0.35, 0.90], dtype=np.float32),  # class 2 — blue
    ]

    num_instances = int(rng.integers(1, max_instances + 1))
    boxes = generate_non_overlapping_boxes(num_instances, image_size, min_box_size, rng)

    targets: list[dict] = []
    for box in boxes:
        x1, y1, x2, y2 = box
        cls_idx = int(rng.integers(0, num_classes))
        colour = class_colours[cls_idx] + rng.uniform(-0.05, 0.05, 3).astype(np.float32)
        colour = np.clip(colour, 0.0, 1.0)

        # Paint rectangle onto image
        for c in range(3):
            image_np[c, y1:y2, x1:x2] = colour[c]

        # Build binary mask
        mask_np = np.zeros((image_size, image_size), dtype=np.float32)
        mask_np[y1:y2, x1:x2] = 1.0

        targets.append({
            "box": torch.tensor([x1, y1, x2, y2], dtype=torch.float32),
            "class": cls_idx,
            "mask": torch.from_numpy(mask_np),
        })

    image = torch.from_numpy(image_np)
    return image, targets


# Generate all splits
def build_dataset(
    num_samples: int,
    image_size: int,
    num_classes: int,
    max_instances: int,
    min_box_size: int,
    seed: int,
) -> list[tuple[torch.Tensor, list[dict]]]:
    """Generate a synthetic instance segmentation dataset.

    Args:
        num_samples: Number of images to generate.
        image_size: Spatial size (H = W) of each image.
        num_classes: Number of foreground classes.
        max_instances: Maximum instances per image.
        min_box_size: Minimum bounding-box side in pixels.
        seed: Random seed for reproducibility.

    Returns:
        List of (image_tensor, targets) pairs.
    """
    rng = np.random.default_rng(seed)
    dataset: list[tuple[torch.Tensor, list[dict]]] = []
    for _ in range(num_samples):
        sample = make_instance_segmentation_sample(
            image_size, num_classes, max_instances, min_box_size, rng
        )
        dataset.append(sample)
    return dataset


train_data = build_dataset(NUM_TRAIN, IMAGE_SIZE, NUM_CLASSES, MAX_INSTANCES, MIN_BOX_SIZE, SEED)
val_data   = build_dataset(NUM_VAL,   IMAGE_SIZE, NUM_CLASSES, MAX_INSTANCES, MIN_BOX_SIZE, SEED + 1)
test_data  = build_dataset(NUM_TEST,  IMAGE_SIZE, NUM_CLASSES, MAX_INSTANCES, MIN_BOX_SIZE, SEED + 2)

print(f"Train: {len(train_data)} images | Val: {len(val_data)} | Test: {len(test_data)}")
instance_counts = [len(t) for _, t in train_data]
print(f"Train instances per image — min: {min(instance_counts)}, "
      f"max: {max(instance_counts)}, mean: {np.mean(instance_counts):.2f}")
class_counter = {0: 0, 1: 0, 2: 0}
for _, targets in train_data:
    for t in targets:
        class_counter[t["class"]] += 1
print(f"Class distribution (train): {class_counter}")

In [ ]:
# ── EDA — visualise sample images with GT boxes and masks ──────────────────
CLASS_NAMES = ["Red", "Green", "Blue"]
MASK_COLOURS = ["Reds", "Greens", "Blues"]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col_idx in range(4):
    img, targets = train_data[col_idx]
    img_np = img.permute(1, 2, 0).numpy()  # (H, W, 3)

    # Top row: raw image with GT boxes
    ax_img = axes[0, col_idx]
    ax_img.imshow(img_np)
    for tgt in targets:
        x1, y1, x2, y2 = tgt["box"].numpy()
        cls_idx = tgt["class"]
        rect = mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=["red", "green", "blue"][cls_idx], facecolor="none"
        )
        ax_img.add_patch(rect)
        ax_img.text(x1, y1 - 2, CLASS_NAMES[cls_idx], color=["red", "green", "blue"][cls_idx],
                    fontsize=8, fontweight="bold")
    ax_img.set_title(f"Image {col_idx} — {len(targets)} instance(s)", fontsize=9)
    ax_img.axis("off")

    # Bottom row: overlay all instance masks
    ax_msk = axes[1, col_idx]
    ax_msk.imshow(img_np)
    combined_mask = np.zeros((*img_np.shape[:2], 4))  # RGBA
    mask_rgba_colours = [
        [1.0, 0.2, 0.2, 0.45],
        [0.2, 1.0, 0.2, 0.45],
        [0.2, 0.2, 1.0, 0.45],
    ]
    for tgt in targets:
        mask = tgt["mask"].numpy()
        rgba = np.array(mask_rgba_colours[tgt["class"]])
        for c in range(4):
            combined_mask[:, :, c] += mask * rgba[c]
    combined_mask = np.clip(combined_mask, 0.0, 1.0)
    ax_msk.imshow(combined_mask)
    ax_msk.set_title(f"Instance masks", fontsize=9)
    ax_msk.axis("off")

plt.suptitle("Synthetic Instance Segmentation Dataset — Sample Images", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 1 — RoI Pooling from Scratch

### 1.1 Why Fixed-Size Features?

In a two-stage detector (e.g., Faster R-CNN), a **region proposal** (bounding box) can have arbitrary size. The classification and mask heads downstream expect a **fixed-size feature vector**. RoI operations extract a fixed $k \times k$ feature grid from any proposed region of the shared feature map.

### 1.2 RoI Pooling — Algorithm

Given a feature map $\mathbf{F} \in \mathbb{R}^{C \times H' \times W'}$ and a box $(x_1, y_1, x_2, y_2)$ in **feature-map coordinates**, RoI Pooling proceeds:

1. Divide the RoI into an $k \times k$ grid of equal-area bins.
2. For each bin, apply **max pooling** over the feature values in that bin.
3. Output: $\mathbf{R} \in \mathbb{R}^{C \times k \times k}$ — same number of channels, fixed spatial size.

The bin boundaries are computed by dividing the box width and height into $k$ equal segments:
$$\text{bin}_{i,j} = \left[x_1 + \frac{j}{k}(x_2 - x_1),\; x_1 + \frac{j+1}{k}(x_2 - x_1)\right] \times \left[y_1 + \frac{i}{k}(y_2 - y_1),\; y_1 + \frac{i+1}{k}(y_2 - y_1)\right]$$

**Key limitation:** Integer quantisation at two points — when mapping the original image box to the feature map, and when dividing into bins. This quantisation introduces misalignment that hurts mask quality.

In [ ]:
def roi_pool(
    feature_map: torch.Tensor,
    boxes: torch.Tensor,
    output_size: int,
) -> torch.Tensor:
    """RoI Pooling — extract fixed-size feature grids via max pooling.

    Implements RoI Pooling from scratch without torchvision. Each box region
    is divided into an output_size × output_size grid and max-pooled per bin.

    Args:
        feature_map: Float tensor of shape (C, H, W) — single image feature map.
        boxes: Float tensor of shape (N, 4) with columns [x1, y1, x2, y2] in
            feature-map pixel coordinates (may be fractional).
        output_size: Number of bins per spatial dimension (k). Output is (N, C, k, k).

    Returns:
        Pooled features of shape (N, C, output_size, output_size).
    """
    num_channels, feat_h, feat_w = feature_map.shape
    num_boxes = boxes.shape[0]
    output = torch.zeros(
        num_boxes, num_channels, output_size, output_size,
        dtype=feature_map.dtype, device=feature_map.device
    )

    for box_idx in range(num_boxes):
        x1, y1, x2, y2 = boxes[box_idx].tolist()

        # Clamp box to feature map boundaries
        x1 = max(0.0, min(x1, feat_w - 1.0))
        y1 = max(0.0, min(y1, feat_h - 1.0))
        x2 = max(x1 + 1.0, min(x2, float(feat_w)))
        y2 = max(y1 + 1.0, min(y2, float(feat_h)))

        roi_w = x2 - x1
        roi_h = y2 - y1
        bin_w = roi_w / output_size
        bin_h = roi_h / output_size

        for row in range(output_size):
            for col in range(output_size):
                # Quantise bin boundaries to integers
                bx1 = int(x1 + col * bin_w)
                by1 = int(y1 + row * bin_h)
                bx2 = int(x1 + (col + 1) * bin_w)
                by2 = int(y1 + (row + 1) * bin_h)

                # Ensure minimum bin size of 1
                bx2 = max(bx1 + 1, bx2)
                by2 = max(by1 + 1, by2)

                # Clamp to valid feature map indices
                bx2 = min(bx2, feat_w)
                by2 = min(by2, feat_h)

                # Slice and max-pool the bin
                region = feature_map[:, by1:by2, bx1:bx2]  # (C, bH, bW)
                pooled = F.adaptive_max_pool2d(region.unsqueeze(0), (1, 1))  # (1, C, 1, 1)
                output[box_idx, :, row, col] = pooled.squeeze()

    return output


# ── Quick sanity check ─────────────────────────────────────────────────────
torch.manual_seed(SEED)
dummy_fmap = torch.randn(8, FEATURE_SIZE, FEATURE_SIZE)  # (C=8, H=16, W=16)
# Two boxes in feature-map coordinates
dummy_boxes = torch.tensor([[1.0, 1.0, 8.0, 8.0], [9.0, 9.0, 15.0, 15.0]])
pooled_roi = roi_pool(dummy_fmap, dummy_boxes, output_size=ROI_OUTPUT_SIZE)
print(f"Input feature map : {tuple(dummy_fmap.shape)}")
print(f"Boxes             : {tuple(dummy_boxes.shape)}")
print(f"RoI Pool output   : {tuple(pooled_roi.shape)}  — expected (2, 8, 7, 7)")

In [ ]:
# ── Visualise RoI Pooling ──────────────────────────────────────────────────
torch.manual_seed(SEED)
vis_fmap = torch.randn(1, FEATURE_SIZE, FEATURE_SIZE)  # single-channel for display
vis_box  = torch.tensor([[2.0, 2.0, 12.0, 10.0]])      # one RoI
vis_pooled = roi_pool(vis_fmap, vis_box, output_size=ROI_OUTPUT_SIZE)  # (1, 1, 7, 7)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Feature map
ax = axes[0]
ax.imshow(vis_fmap[0].numpy(), cmap="viridis")
x1, y1, x2, y2 = vis_box[0].tolist()
rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                            linewidth=2, edgecolor="red", facecolor="none")
ax.add_patch(rect)
ax.set_title(f"Feature Map ({FEATURE_SIZE}×{FEATURE_SIZE})\nRed box = GT RoI", fontsize=10)
ax.set_xlabel("Width"); ax.set_ylabel("Height")

# Zoom into RoI region
ax2 = axes[1]
roi_region = vis_fmap[0, int(y1):int(y2), int(x1):int(x2)]
ax2.imshow(roi_region.numpy(), cmap="viridis")
ax2.set_title(f"RoI Region Crop\n({int(y2-y1)}×{int(x2-x1)} pixels)", fontsize=10)
ax2.set_xlabel("Width"); ax2.set_ylabel("Height")

# Pooled output
ax3 = axes[2]
ax3.imshow(vis_pooled[0, 0].numpy(), cmap="viridis")
ax3.set_title(f"RoI Pool Output\n({ROI_OUTPUT_SIZE}×{ROI_OUTPUT_SIZE} — fixed size)", fontsize=10)
ax3.set_xlabel("Width"); ax3.set_ylabel("Height")

plt.suptitle("RoI Pooling: Feature Map → Fixed-Size Feature Grid", fontsize=12)
plt.tight_layout()
plt.show()
print("Spatial extent captured: each output cell aggregates one bin of the RoI via max pooling.")

---
## Part 2 — RoI Align from Scratch

### 2.1 The Quantisation Problem

RoI Pooling introduces two rounding errors:
1. **Box-to-feature-map mapping:** The original image box coordinates are divided by the stride and **rounded** to feature-map integers.
2. **Bin boundaries:** Each bin boundary $x_1 + j \cdot \text{bin\_w}$ is **rounded** to the nearest integer before slicing.

For masks these errors are catastrophic — a single pixel misalignment at the feature level can translate to 8–32 pixel errors in the output mask at high resolution.

### 2.2 RoI Align — Algorithm

RoI Align eliminates both rounding steps by using **bilinear interpolation** at exact sub-pixel locations:

1. Divide the RoI into a $k \times k$ grid — **without rounding** bin boundaries.
2. Inside each bin, place a regular grid of $r \times r$ **sampling points** (where $r$ = `sampling_ratio`).
3. For each sampling point $(y_s, x_s)$, compute the feature value via **bilinear interpolation** from the four surrounding integer grid points.
4. **Average** the $r^2$ sample values to produce the bin's output.

Bilinear interpolation at sub-pixel point $(x, y)$:
$$f(x, y) = (1-\alpha)(1-\beta)\,f_{\lfloor x \rfloor, \lfloor y \rfloor} + \alpha(1-\beta)\,f_{\lceil x \rceil, \lfloor y \rfloor} + (1-\alpha)\beta\,f_{\lfloor x \rfloor, \lceil y \rceil} + \alpha\beta\,f_{\lceil x \rceil, \lceil y \rceil}$$
where $\alpha = x - \lfloor x \rfloor$ and $\beta = y - \lfloor y \rfloor$.

In [ ]:
def bilinear_interpolate(
    feature_map: torch.Tensor,
    y: float,
    x: float,
) -> torch.Tensor:
    """Bilinear interpolation at a sub-pixel location in a feature map.

    Samples from the four surrounding integer grid points using bilinear
    weights derived from the fractional offsets. Out-of-bounds coordinates
    are clamped to the valid feature map extent before sampling.

    Args:
        feature_map: Float tensor of shape (C, H, W).
        y: Sub-pixel row coordinate (may be fractional).
        x: Sub-pixel column coordinate (may be fractional).

    Returns:
        Interpolated feature vector of shape (C,).
    """
    num_channels, feat_h, feat_w = feature_map.shape

    # Integer floor/ceil neighbours
    y0 = int(max(0, min(feat_h - 1, int(y))))
    x0 = int(max(0, min(feat_w - 1, int(x))))
    y1 = int(max(0, min(feat_h - 1, y0 + 1)))
    x1 = int(max(0, min(feat_w - 1, x0 + 1)))

    # Fractional offsets (bilinear weights)
    alpha = x - int(x)   # horizontal fraction
    beta  = y - int(y)   # vertical fraction

    # Weighted combination of four corners
    top_left     = feature_map[:, y0, x0]   # (C,)
    top_right    = feature_map[:, y0, x1]
    bottom_left  = feature_map[:, y1, x0]
    bottom_right = feature_map[:, y1, x1]

    interpolated = (
        (1 - alpha) * (1 - beta) * top_left
        + alpha     * (1 - beta) * top_right
        + (1 - alpha) * beta     * bottom_left
        + alpha       * beta     * bottom_right
    )  # (C,)
    return interpolated


def roi_align(
    feature_map: torch.Tensor,
    boxes: torch.Tensor,
    output_size: int,
    sampling_ratio: int = 2,
) -> torch.Tensor:
    """RoI Align — extract fixed-size features via bilinear interpolation.

    Implements RoI Align from scratch (no torchvision). Eliminates the
    quantisation error of RoI Pooling by using sub-pixel sampling points
    and bilinear interpolation within each bin.

    Args:
        feature_map: Float tensor of shape (C, H, W) — single image feature map.
        boxes: Float tensor of shape (N, 4) with columns [x1, y1, x2, y2] in
            feature-map coordinates (may be fractional).
        output_size: Number of bins per spatial dimension (k). Output is (N, C, k, k).
        sampling_ratio: Number of sampling points per bin side. Total points
            per bin = sampling_ratio². Defaults to 2.

    Returns:
        Aligned features of shape (N, C, output_size, output_size).
    """
    num_channels, feat_h, feat_w = feature_map.shape
    num_boxes = boxes.shape[0]
    output = torch.zeros(
        num_boxes, num_channels, output_size, output_size,
        dtype=feature_map.dtype, device=feature_map.device
    )

    for box_idx in range(num_boxes):
        x1, y1, x2, y2 = boxes[box_idx].tolist()

        # No quantisation — use exact float coordinates
        roi_w = x2 - x1
        roi_h = y2 - y1

        # Bin dimensions (exact, no rounding)
        bin_w = roi_w / output_size
        bin_h = roi_h / output_size

        for row in range(output_size):
            for col in range(output_size):
                # Exact bin boundaries
                bin_x1 = x1 + col * bin_w
                bin_y1 = y1 + row * bin_h

                # Place sampling_ratio × sampling_ratio points inside the bin
                sample_values = torch.zeros(num_channels, device=feature_map.device)
                for sy in range(sampling_ratio):
                    for sx in range(sampling_ratio):
                        # Sample point at centre of sub-cell
                        sample_x = bin_x1 + (sx + 0.5) * bin_w / sampling_ratio
                        sample_y = bin_y1 + (sy + 0.5) * bin_h / sampling_ratio

                        # Bilinear interpolation at (sample_y, sample_x)
                        interp = bilinear_interpolate(feature_map, sample_y, sample_x)
                        sample_values = sample_values + interp

                # Average over all sampling points
                output[box_idx, :, row, col] = sample_values / (sampling_ratio ** 2)

    return output


# ── Sanity check ──────────────────────────────────────────────────────────
aligned_roi = roi_align(dummy_fmap, dummy_boxes, output_size=ROI_OUTPUT_SIZE, sampling_ratio=SAMPLING_RATIO)
print(f"RoI Align output : {tuple(aligned_roi.shape)}  — expected (2, 8, 7, 7)")

In [ ]:
# ── Quantitative comparison: RoI Pool vs RoI Align alignment error ─────────
# We create a known feature map (smooth gradient) where we know the ground
# truth value at every sub-pixel location.

torch.manual_seed(SEED)
C_compare = 4
H_compare = 32

# Smooth gradient feature map: value at (c, y, x) = c * y * x / (H*W)
ys = torch.linspace(0, 1, H_compare)
xs = torch.linspace(0, 1, H_compare)
grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
gradient_fmap = torch.stack([
    (c + 1) * grid_y * grid_x for c in range(C_compare)
])  # (C, H, W)

# Test a slightly sub-pixel box (non-integer boundaries)
test_box = torch.tensor([[1.7, 2.3, 20.5, 19.8]])
k_compare = 5  # output size

pool_out  = roi_pool(gradient_fmap, test_box, output_size=k_compare)
align_out = roi_align(gradient_fmap, test_box, output_size=k_compare, sampling_ratio=2)

# Ground truth: use RoI Align with very dense sampling as reference
gt_out = roi_align(gradient_fmap, test_box, output_size=k_compare, sampling_ratio=8)

pool_error  = (pool_out  - gt_out).pow(2).mean().sqrt().item()
align_error = (align_out - gt_out).pow(2).mean().sqrt().item()

print("Alignment Error (RMSE vs high-precision RoI Align reference):")
print(f"  RoI Pooling : {pool_error:.6f}")
print(f"  RoI Align   : {align_error:.6f}")
print(f"  Improvement : {pool_error / (align_error + 1e-9):.1f}× lower error with RoI Align")

In [ ]:
# ── Visual side-by-side: Pool vs Align on the gradient feature map ─────────
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

# Reference: high-res align
axes[0].imshow(gt_out[0, 0].numpy(), cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("GT (RoI Align, r=8)", fontsize=10)
axes[0].axis("off")

axes[1].imshow(pool_out[0, 0].numpy(), cmap="viridis", vmin=0, vmax=1)
axes[1].set_title(f"RoI Pooling\nRMSE={pool_error:.4f}", fontsize=10)
axes[1].axis("off")

axes[2].imshow(align_out[0, 0].numpy(), cmap="viridis", vmin=0, vmax=1)
axes[2].set_title(f"RoI Align (r=2)\nRMSE={align_error:.4f}", fontsize=10)
axes[2].axis("off")

# Error map
pool_err_map  = (pool_out[0, 0]  - gt_out[0, 0]).abs().numpy()
align_err_map = (align_out[0, 0] - gt_out[0, 0]).abs().numpy()
combined_err  = np.stack([pool_err_map, align_err_map], axis=0)
vmax_err = combined_err.max() + 1e-8

im = axes[3].imshow(pool_err_map, cmap="coolwarm", vmin=0, vmax=vmax_err)
axes[3].set_title("Pool Absolute Error\n(vs GT)", fontsize=10)
axes[3].axis("off")
plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

plt.suptitle("RoI Pooling vs RoI Align — Alignment Quality on Gradient Feature Map", fontsize=11)
plt.tight_layout()
plt.show()

---
## Part 3 — Mask Head

### 3.1 Architecture

The **Mask Head** takes a fixed-size RoI feature grid $(C \times k \times k)$ and predicts a binary segmentation mask for the **ground-truth class only** (Mask R-CNN convention). This decoupling from the classification head means the mask branch focuses solely on shape, not class recognition.

Architecture:
- 4 × Conv2d + ReLU (stride=1, padding=1) — maintain spatial resolution, increase channel depth.
- ConvTranspose2d (stride=2) — upsample from $k \times k$ to $2k \times 2k$.
- Conv2d → `num_classes` channels — one binary logit channel per class.

**Loss:** Binary cross-entropy applied **only to the GT class channel**:
$$\mathcal{L}_{\text{mask}} = \frac{1}{N} \sum_{i=1}^{N} \text{BCE}(\hat{m}_i^{(c_i)},\, m_i)$$
where $c_i$ is the class of the $i$-th instance and $m_i \in \{0, 1\}^{H_m \times W_m}$ is the GT binary mask resized to the head's output resolution.

In [ ]:
class MaskHead(nn.Module):
    """Mask prediction head — predicts per-class binary segmentation masks.

    Takes a batch of fixed-size RoI features and outputs one binary mask
    channel per class. During loss computation only the GT class channel
    is supervised (Mask R-CNN convention).

    Attributes:
        conv_layers: Sequential 4-layer Conv + ReLU feature refiner.
        upsample: ConvTranspose2d that doubles spatial resolution.
        mask_logits: Final 1×1 Conv producing per-class logit maps.
    """

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_dim: int = 64,
    ) -> None:
        """Initialise MaskHead layers.

        Args:
            in_channels: Number of input feature channels (from backbone).
            num_classes: Number of foreground classes. One output channel per class.
            hidden_dim: Number of channels used in the intermediate conv layers.
        """
        super().__init__()
        # 4 conv layers maintain spatial resolution
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        # Upsample × 2
        self.upsample = nn.ConvTranspose2d(
            hidden_dim, hidden_dim, kernel_size=2, stride=2
        )
        # One logit map per class
        self.mask_logits = nn.Conv2d(hidden_dim, num_classes, kernel_size=1)

    def forward(self, roi_features: torch.Tensor) -> torch.Tensor:
        """Forward pass through the mask head.

        Args:
            roi_features: Float tensor of shape (N, C, k, k) — RoI-aligned features.

        Returns:
            Mask logits of shape (N, num_classes, 2k, 2k). Apply sigmoid for probabilities.
        """
        features = self.conv_layers(roi_features)       # (N, hidden_dim, k, k)
        features = F.relu(self.upsample(features))      # (N, hidden_dim, 2k, 2k)
        logits   = self.mask_logits(features)           # (N, num_classes, 2k, 2k)
        return logits


class MaskLoss(nn.Module):
    """Binary mask loss — supervises only the GT class channel.

    Follows the Mask R-CNN convention: instead of predicting a single binary
    mask regardless of class, the head outputs one binary mask per class and
    we only compute loss on the predicted channel that corresponds to the GT class.
    This prevents competition between classes during mask learning.

    Attributes:
        bce_loss: Binary cross-entropy with logits loss module.
    """

    def __init__(self) -> None:
        """Initialise the mask loss with binary cross-entropy."""
        super().__init__()
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(
        self,
        mask_logits: torch.Tensor,
        gt_masks: torch.Tensor,
        gt_classes: torch.Tensor,
    ) -> torch.Tensor:
        """Compute binary mask loss over a batch of proposals.

        Args:
            mask_logits: Float tensor of shape (N, num_classes, Hm, Wm).
            gt_masks: Float binary tensor of shape (N, Hm, Wm) — ground-truth masks
                resized to the mask head output resolution.
            gt_classes: Long tensor of shape (N,) — GT class index for each proposal.

        Returns:
            Scalar loss tensor.
        """
        num_instances = mask_logits.shape[0]
        # Select the GT class channel for each instance
        selected_logits = mask_logits[
            torch.arange(num_instances, device=mask_logits.device),
            gt_classes
        ]  # (N, Hm, Wm)
        return self.bce_loss(selected_logits, gt_masks)


# ── Sanity check mask head ────────────────────────────────────────────────
torch.manual_seed(SEED)
mask_head = MaskHead(in_channels=BACKBONE_CHANNELS, num_classes=NUM_CLASSES, hidden_dim=HIDDEN_DIM)
mask_criterion = MaskLoss()

# Fake RoI features: 3 proposals, 64 channels, 7×7
fake_roi = torch.randn(3, BACKBONE_CHANNELS, ROI_OUTPUT_SIZE, ROI_OUTPUT_SIZE)
fake_mask_logits = mask_head(fake_roi)  # (3, NUM_CLASSES, 14, 14)
print(f"Mask head input  : {tuple(fake_roi.shape)}")
print(f"Mask head output : {tuple(fake_mask_logits.shape)}  — (N, num_classes, 2k, 2k)")

# Fake GT masks and classes
fake_gt_masks   = (torch.rand(3, MASK_SIZE, MASK_SIZE) > 0.5).float()
fake_gt_classes = torch.tensor([0, 1, 2], dtype=torch.long)
mask_loss_val   = mask_criterion(fake_mask_logits, fake_gt_masks, fake_gt_classes)
print(f"Mask loss (random init): {mask_loss_val.item():.4f}")

In [ ]:
# ── Visualise mask predictions before and after 5 training steps ──────────
torch.manual_seed(SEED)
demo_mask_head = MaskHead(in_channels=BACKBONE_CHANNELS, num_classes=NUM_CLASSES, hidden_dim=HIDDEN_DIM)
demo_optimizer = torch.optim.Adam(demo_mask_head.parameters(), lr=LEARNING_RATE)
demo_criterion = MaskLoss()

# Sample 3 instances from training data for the demo
demo_images_list = []
demo_masks_list  = []
demo_classes_list = []
for img, targets in train_data[:5]:
    if len(targets) > 0:
        tgt = targets[0]
        demo_masks_list.append(tgt["mask"])
        demo_classes_list.append(tgt["class"])
        if len(demo_masks_list) == 3:
            break

# Create fake fixed RoI features for the demo
torch.manual_seed(SEED)
demo_roi_feats = torch.randn(3, BACKBONE_CHANNELS, ROI_OUTPUT_SIZE, ROI_OUTPUT_SIZE)
demo_gt_masks_resized = torch.stack([
    F.interpolate(m.unsqueeze(0).unsqueeze(0), size=(MASK_SIZE, MASK_SIZE), mode="nearest").squeeze()
    for m in demo_masks_list
])  # (3, MASK_SIZE, MASK_SIZE)
demo_gt_classes = torch.tensor(demo_classes_list, dtype=torch.long)

# Record predictions before training
with torch.no_grad():
    pred_before = torch.sigmoid(demo_mask_head(demo_roi_feats))  # (3, C, 14, 14)

# 5 gradient steps
for _step in range(5):
    demo_optimizer.zero_grad()
    logits = demo_mask_head(demo_roi_feats)
    loss = demo_criterion(logits, demo_gt_masks_resized, demo_gt_classes)
    loss.backward()
    demo_optimizer.step()

with torch.no_grad():
    pred_after = torch.sigmoid(demo_mask_head(demo_roi_feats))  # (3, C, 14, 14)

# Visualise
fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for i in range(3):
    cls_i = demo_gt_classes[i].item()
    gt_mask_vis  = demo_gt_masks_resized[i].numpy()
    before_vis   = pred_before[i, cls_i].numpy()
    after_vis    = pred_after[i, cls_i].numpy()
    diff_vis     = after_vis - before_vis

    axes[i, 0].imshow(gt_mask_vis, cmap="gray", vmin=0, vmax=1)
    axes[i, 0].set_title(f"GT Mask (inst {i}, cls {cls_i})", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(before_vis, cmap="viridis", vmin=0, vmax=1)
    axes[i, 1].set_title("Pred (before training)", fontsize=9)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(after_vis, cmap="viridis", vmin=0, vmax=1)
    axes[i, 2].set_title("Pred (after 5 steps)", fontsize=9)
    axes[i, 2].axis("off")

    im = axes[i, 3].imshow(diff_vis, cmap="coolwarm", vmin=-0.5, vmax=0.5)
    axes[i, 3].set_title("Δ (after − before)", fontsize=9)
    axes[i, 3].axis("off")
    plt.colorbar(im, ax=axes[i, 3], fraction=0.046, pad=0.04)

plt.suptitle("Mask Head: Predictions Before vs After 5 Gradient Steps", fontsize=11)
plt.tight_layout()
plt.show()

---
## Part 4 — Full InstanceSegmentationModel + Training

### 4.1 Pipeline Architecture

We now assemble all components into a toy Mask R-CNN–style pipeline:

```
Image (3, 128, 128)
    │
    ▼  ToyBackbone (3 conv blocks, stride 8 total)
Feature Map (64, 16, 16)
    │
    ▼  ProposalLayer — uses GT boxes scaled to feature coords
Proposals in feature coords (N_prop, 4)
    │
    ▼  RoI Align  (output_size=7)
RoI Features (N_prop, 64, 7, 7)
    │
    ├──▶  Classification Head (Linear) → cls logits (N_prop, num_classes+1)
    ├──▶  Box Regression Head (Linear) → box deltas  (N_prop, 4)
    └──▶  Mask Head (4 conv + upsample) → mask logits (N_prop, num_classes, 14, 14)
```

**Training shortcut:** Instead of training a full Region Proposal Network (RPN), we use GT boxes directly as proposals. This **oracle proposal** strategy lets us focus on the RoI Align + Mask Head learning, which is the subject of this notebook. A real RPN is introduced in the detection notebooks (09-01, 09-02).

**Loss:** Combined total loss:
$$\mathcal{L} = \mathcal{L}_{\text{cls}} + \lambda_{\text{reg}} \cdot \mathcal{L}_{\text{reg}} + \lambda_{\text{mask}} \cdot \mathcal{L}_{\text{mask}}$$
with $\lambda_{\text{reg}} = \lambda_{\text{mask}} = 1$.

In [ ]:
class ToyBackbone(nn.Module):
    """Lightweight CNN backbone producing a single-scale feature map.

    Uses three conv blocks each with stride 2, yielding a total stride of 8.
    For a 128×128 input, the output feature map is 64 channels × 16×16.

    Attributes:
        block1: First conv block (stride 2) — 3 → 32 channels.
        block2: Second conv block (stride 2) — 32 → 64 channels.
        block3: Third conv block (stride 2) — 64 → 64 channels.
    """

    def __init__(self, in_channels: int = 3, out_channels: int = 64) -> None:
        """Initialise the backbone with three strided conv blocks.

        Args:
            in_channels: Number of input image channels (e.g., 3 for RGB).
            out_channels: Number of output feature map channels.
        """
        super().__init__()
        mid_channels = out_channels // 2
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Extract features from a batch of images.

        Args:
            images: Float tensor of shape (B, C, H, W).

        Returns:
            Feature map of shape (B, out_channels, H//8, W//8).
        """
        x = self.block1(images)
        x = self.block2(x)
        x = self.block3(x)
        return x


class InstanceSegmentationModel(nn.Module):
    """Toy Mask R-CNN–style instance segmentation model.

    Combines a CNN backbone, RoI Align feature extraction, and three parallel
    prediction heads (classification, box regression, mask) into a single
    end-to-end model. Uses GT boxes as proposals during training (oracle mode).

    Attributes:
        backbone: Shared feature extractor (ToyBackbone).
        cls_head: Linear layer mapping flattened RoI features to class logits.
        reg_head: Linear layer mapping flattened RoI features to box deltas.
        mask_head: MaskHead for binary mask prediction.
        feature_stride: Ratio of input resolution to feature map resolution.
        roi_output_size: Spatial size k for RoI Align output (k × k).
        num_classes: Number of foreground classes.
    """

    def __init__(
        self,
        in_channels: int,
        backbone_channels: int,
        num_classes: int,
        roi_output_size: int,
        mask_hidden_dim: int,
        fc_dim: int,
        feature_stride: int,
        sampling_ratio: int = 2,
    ) -> None:
        """Initialise all sub-modules of the instance segmentation model.

        Args:
            in_channels: Number of input image channels.
            backbone_channels: Number of channels output by the backbone.
            num_classes: Number of foreground classes (background is implicit).
            roi_output_size: Spatial size k for RoI Align output.
            mask_hidden_dim: Hidden channels in the mask head conv layers.
            fc_dim: Fully connected layer width for classification / regression.
            feature_stride: Image-to-feature-map stride (e.g., 8 for three stride-2 blocks).
            sampling_ratio: Number of RoI Align sample points per bin side.
        """
        super().__init__()
        self.backbone        = ToyBackbone(in_channels, backbone_channels)
        self.feature_stride  = feature_stride
        self.roi_output_size = roi_output_size
        self.num_classes     = num_classes
        self.sampling_ratio  = sampling_ratio

        flat_dim = backbone_channels * roi_output_size * roi_output_size
        # Classification head: num_classes foreground + 1 background
        self.cls_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes + 1),
        )
        # Box regression head: 4 deltas per class (or class-agnostic → 4)
        self.reg_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, 4),
        )
        self.mask_head = MaskHead(
            in_channels=backbone_channels,
            num_classes=num_classes,
            hidden_dim=mask_hidden_dim,
        )

    def extract_roi_features(
        self,
        feature_map: torch.Tensor,
        boxes_image_coords: list[torch.Tensor],
    ) -> tuple[torch.Tensor, list[int]]:
        """Run RoI Align on all proposals across a batch.

        Scales box coordinates from image space to feature-map space and
        applies RoI Align per image, then concatenates across the batch.

        Args:
            feature_map: Batch feature map of shape (B, C, H', W').
            boxes_image_coords: List of B tensors, each (N_i, 4), giving GT box
                coordinates in image pixel space [x1, y1, x2, y2].

        Returns:
            Tuple of:
              - roi_features: Concatenated RoI features (sum_N_i, C, k, k).
              - instance_counts: List of N_i values (one per image in batch).
        """
        scale = 1.0 / self.feature_stride
        all_roi_features: list[torch.Tensor] = []
        instance_counts: list[int] = []

        for batch_idx in range(feature_map.shape[0]):
            fmap_i = feature_map[batch_idx]  # (C, H', W')
            boxes_i = boxes_image_coords[batch_idx]  # (N_i, 4)

            if boxes_i.shape[0] == 0:
                instance_counts.append(0)
                continue

            # Scale image-coord boxes to feature-map coords
            feat_boxes_i = boxes_i * scale  # (N_i, 4)
            roi_feats_i  = roi_align(
                fmap_i, feat_boxes_i,
                output_size=self.roi_output_size,
                sampling_ratio=self.sampling_ratio,
            )  # (N_i, C, k, k)
            all_roi_features.append(roi_feats_i)
            instance_counts.append(boxes_i.shape[0])

        if len(all_roi_features) == 0:
            dummy = torch.zeros(
                0, feature_map.shape[1],
                self.roi_output_size, self.roi_output_size,
                device=feature_map.device
            )
            return dummy, instance_counts

        return torch.cat(all_roi_features, dim=0), instance_counts

    def forward(
        self,
        images: torch.Tensor,
        proposals: list[torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Forward pass — extract features and predict class, box, and mask.

        Args:
            images: Float tensor of shape (B, C, H, W).
            proposals: List of B tensors, each (N_i, 4), GT boxes in image coords.

        Returns:
            Tuple of:
              - cls_logits:  (sum_N_i, num_classes+1) classification logits.
              - box_preds:   (sum_N_i, 4) box delta predictions.
              - mask_logits: (sum_N_i, num_classes, 2k, 2k) mask logits.
        """
        feature_map = self.backbone(images)                              # (B, C, H', W')
        roi_features, _ = self.extract_roi_features(feature_map, proposals)  # (N, C, k, k)

        cls_logits  = self.cls_head(roi_features)    # (N, num_classes+1)
        box_preds   = self.reg_head(roi_features)    # (N, 4)
        mask_logits = self.mask_head(roi_features)   # (N, num_classes, 2k, 2k)
        return cls_logits, box_preds, mask_logits

    def compute_loss(
        self,
        cls_logits: torch.Tensor,
        box_preds: torch.Tensor,
        mask_logits: torch.Tensor,
        targets: list[list[dict]],
        mask_size: int,
        device: torch.device,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Compute combined instance segmentation loss.

        Aggregates GT labels from all instances in the batch and computes:
          - Classification loss (cross-entropy over foreground classes).
          - Regression loss (smooth L1 on box deltas, treated as zero here
            because GT boxes ARE the proposals).
          - Mask loss (binary CE on GT class channel only).

        Args:
            cls_logits: (sum_N_i, num_classes+1) classification logits.
            box_preds: (sum_N_i, 4) predicted box deltas.
            mask_logits: (sum_N_i, num_classes, Hm, Wm) mask logits.
            targets: Batch of target lists. Each element is a list of dicts with
                keys 'box', 'class', 'mask'.
            mask_size: Spatial size to resize GT masks before loss.
            device: Compute device.

        Returns:
            Tuple (total_loss, cls_loss, reg_loss, mask_loss) — all scalar tensors.
        """
        # Aggregate GT labels across batch
        gt_classes_list: list[int] = []
        gt_masks_list:   list[torch.Tensor] = []
        gt_boxes_list:   list[torch.Tensor] = []

        for sample_targets in targets:
            for tgt in sample_targets:
                gt_classes_list.append(tgt["class"])
                gt_boxes_list.append(tgt["box"])
                resized_mask = F.interpolate(
                    tgt["mask"].unsqueeze(0).unsqueeze(0),
                    size=(mask_size, mask_size),
                    mode="nearest",
                ).squeeze()  # (mask_size, mask_size)
                gt_masks_list.append(resized_mask)

        if len(gt_classes_list) == 0:
            zero = torch.tensor(0.0, device=device, requires_grad=True)
            return zero, zero, zero, zero

        gt_classes = torch.tensor(gt_classes_list, dtype=torch.long, device=device)
        gt_masks   = torch.stack(gt_masks_list).to(device)    # (N, mask_size, mask_size)
        gt_boxes   = torch.stack(gt_boxes_list).to(device)    # (N, 4)

        # Classification loss: foreground class labels (no background negatives in oracle mode)
        cls_loss = F.cross_entropy(cls_logits, gt_classes)

        # Box regression: GT boxes are the proposals, so GT delta = 0
        zero_deltas = torch.zeros_like(box_preds)
        reg_loss = F.smooth_l1_loss(box_preds, zero_deltas)

        # Mask loss: BCEWithLogits on GT class channel only
        num_instances = mask_logits.shape[0]
        selected_mask_logits = mask_logits[
            torch.arange(num_instances, device=device), gt_classes
        ]  # (N, mask_size, mask_size)
        mask_loss = F.binary_cross_entropy_with_logits(selected_mask_logits, gt_masks)

        total_loss = cls_loss + reg_loss + mask_loss
        return total_loss, cls_loss, reg_loss, mask_loss


# ── Instantiate and quick forward-pass sanity check ────────────────────────
torch.manual_seed(SEED)
model = InstanceSegmentationModel(
    in_channels=IN_CHANNELS,
    backbone_channels=BACKBONE_CHANNELS,
    num_classes=NUM_CLASSES,
    roi_output_size=ROI_OUTPUT_SIZE,
    mask_hidden_dim=HIDDEN_DIM,
    fc_dim=FC_DIM,
    feature_stride=8,
    sampling_ratio=SAMPLING_RATIO,
).to(device)

# Use first 2 training samples for sanity check
sanity_imgs = torch.stack([train_data[i][0] for i in range(2)]).to(device)
sanity_props = [train_data[i][1] for i in range(2)]
sanity_boxes = [torch.stack([t["box"] for t in props]).to(device) for props in sanity_props]

with torch.no_grad():
    cls_out, box_out, msk_out = model(sanity_imgs, sanity_boxes)

total_instances = sum(len(p) for p in sanity_props)
print(f"Images shape      : {tuple(sanity_imgs.shape)}")
print(f"Total instances   : {total_instances}")
print(f"cls_logits shape  : {tuple(cls_out.shape)}  — (N, {NUM_CLASSES+1})")
print(f"box_preds shape   : {tuple(box_out.shape)}  — (N, 4)")
print(f"mask_logits shape : {tuple(msk_out.shape)}  — (N, {NUM_CLASSES}, {MASK_SIZE}, {MASK_SIZE})")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters  : {total_params:,}")

### 4.2 Custom Dataset and DataLoader

Because each image has a variable number of instances, we cannot use the default PyTorch collate function (it expects tensors of equal size). We write a custom `collate_fn` that keeps images in a batched tensor but leaves targets as a Python list.

In [ ]:
class InstanceSegDataset(Dataset):
    """PyTorch Dataset wrapping the synthetic instance segmentation data.

    Each item is an (image, targets) pair where targets is a list of dicts
    with keys 'box', 'class', and 'mask'.

    Attributes:
        samples: List of (image_tensor, targets_list) pairs.
    """

    def __init__(self, samples: list[tuple[torch.Tensor, list[dict]]]) -> None:
        """Initialise the dataset from pre-generated samples.

        Args:
            samples: List of (image, targets) pairs as returned by build_dataset.
        """
        self.samples = samples

    def __len__(self) -> int:
        """Return the total number of samples in the dataset."""
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, list[dict]]:
        """Retrieve a single (image, targets) pair by index.

        Args:
            idx: Sample index.

        Returns:
            Tuple of (image tensor (3, H, W), list of target dicts).
        """
        return self.samples[idx]


def collate_instance_seg(
    batch: list[tuple[torch.Tensor, list[dict]]],
) -> tuple[torch.Tensor, list[list[dict]]]:
    """Custom collate for variable-length instance segmentation batches.

    Stacks images into a single (B, C, H, W) tensor while keeping targets as
    a Python list (one list of dicts per image).

    Args:
        batch: List of (image, targets) tuples from __getitem__.

    Returns:
        Tuple of:
          - images: Float tensor of shape (B, C, H, W).
          - targets: List of B target-lists.
    """
    images  = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_dataset = InstanceSegDataset(train_data)
val_dataset   = InstanceSegDataset(val_data)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_instance_seg,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_instance_seg,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)

print(f"Train loader: {len(train_loader)} batches × {BATCH_SIZE} images")
print(f"Val loader  : {len(val_loader)} batches × {BATCH_SIZE} images")

### 4.3 Training and Evaluation Functions

In [ ]:
def train_one_epoch(
    model: InstanceSegmentationModel,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    mask_size: int,
) -> tuple[float, float, float, float]:
    """Train the instance segmentation model for one epoch.

    Iterates over the dataloader, runs forward passes, computes the combined
    loss, and updates model parameters via backpropagation.

    Args:
        model: The InstanceSegmentationModel to train.
        dataloader: Training DataLoader (custom collate, variable targets).
        optimizer: Optimizer instance (e.g., Adam).
        device: Compute device.
        mask_size: Spatial size for GT mask resizing in the loss function.

    Returns:
        Tuple (avg_total_loss, avg_cls_loss, avg_reg_loss, avg_mask_loss)
        averaged over all instances in the epoch.
    """
    model.train()
    total_loss_sum = 0.0
    cls_loss_sum   = 0.0
    reg_loss_sum   = 0.0
    mask_loss_sum  = 0.0
    num_instances_seen = 0

    for images, targets in dataloader:
        images = images.to(device)
        # Build list of box tensors on device
        proposals = [
            torch.stack([t["box"] for t in sample_targets]).to(device)
            if len(sample_targets) > 0
            else torch.zeros(0, 4, device=device)
            for sample_targets in targets
        ]

        optimizer.zero_grad()
        cls_logits, box_preds, mask_logits = model(images, proposals)

        total_loss, cls_loss, reg_loss, mask_loss = model.compute_loss(
            cls_logits, box_preds, mask_logits, targets, mask_size, device
        )
        total_loss.backward()
        optimizer.step()

        batch_n = sum(len(t) for t in targets)
        total_loss_sum += total_loss.item() * batch_n
        cls_loss_sum   += cls_loss.item()   * batch_n
        reg_loss_sum   += reg_loss.item()   * batch_n
        mask_loss_sum  += mask_loss.item()  * batch_n
        num_instances_seen += batch_n

    denom = max(num_instances_seen, 1)
    return (
        total_loss_sum / denom,
        cls_loss_sum   / denom,
        reg_loss_sum   / denom,
        mask_loss_sum  / denom,
    )


def evaluate(
    model: InstanceSegmentationModel,
    dataloader: DataLoader,
    device: torch.device,
    mask_size: int,
) -> tuple[float, float, float, float]:
    """Evaluate the model on a validation or test DataLoader.

    Runs inference without gradient computation and accumulates the combined
    loss components across all batches.

    Args:
        model: The InstanceSegmentationModel to evaluate.
        dataloader: Validation or test DataLoader.
        device: Compute device.
        mask_size: Spatial size for GT mask resizing in the loss function.

    Returns:
        Tuple (avg_total_loss, avg_cls_loss, avg_reg_loss, avg_mask_loss).
    """
    model.eval()
    total_loss_sum = 0.0
    cls_loss_sum   = 0.0
    reg_loss_sum   = 0.0
    mask_loss_sum  = 0.0
    num_instances_seen = 0

    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)
            proposals = [
                torch.stack([t["box"] for t in sample_targets]).to(device)
                if len(sample_targets) > 0
                else torch.zeros(0, 4, device=device)
                for sample_targets in targets
            ]
            cls_logits, box_preds, mask_logits = model(images, proposals)

            total_loss, cls_loss, reg_loss, mask_loss = model.compute_loss(
                cls_logits, box_preds, mask_logits, targets, mask_size, device
            )
            batch_n = sum(len(t) for t in targets)
            total_loss_sum += total_loss.item() * batch_n
            cls_loss_sum   += cls_loss.item()   * batch_n
            reg_loss_sum   += reg_loss.item()   * batch_n
            mask_loss_sum  += mask_loss.item()  * batch_n
            num_instances_seen += batch_n

    denom = max(num_instances_seen, 1)
    return (
        total_loss_sum / denom,
        cls_loss_sum   / denom,
        reg_loss_sum   / denom,
        mask_loss_sum  / denom,
    )

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────
torch.manual_seed(SEED)
model = InstanceSegmentationModel(
    in_channels=IN_CHANNELS,
    backbone_channels=BACKBONE_CHANNELS,
    num_classes=NUM_CLASSES,
    roi_output_size=ROI_OUTPUT_SIZE,
    mask_hidden_dim=HIDDEN_DIM,
    fc_dim=FC_DIM,
    feature_stride=8,
    sampling_ratio=SAMPLING_RATIO,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# History lists — standard naming convention
train_losses: list[float] = []
val_losses:   list[float] = []
train_cls_losses:  list[float] = []
train_reg_losses:  list[float] = []
train_mask_losses: list[float] = []
val_cls_losses:    list[float] = []
val_mask_losses:   list[float] = []

best_val_loss   = float("inf")
best_model_state = None

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    train_total, train_cls, train_reg, train_mask = train_one_epoch(
        model, train_loader, optimizer, device, mask_size=MASK_SIZE
    )
    val_total, val_cls, val_reg, val_mask = evaluate(
        model, val_loader, device, mask_size=MASK_SIZE
    )

    elapsed = time.time() - epoch_start
    train_losses.append(train_total)
    val_losses.append(val_total)
    train_cls_losses.append(train_cls)
    train_reg_losses.append(train_reg)
    train_mask_losses.append(train_mask)
    val_cls_losses.append(val_cls)
    val_mask_losses.append(val_mask)

    if val_total < best_val_loss:
        best_val_loss  = val_total
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_total:.4f} "
        f"(cls={train_cls:.4f} reg={train_reg:.4f} mask={train_mask:.4f}) | "
        f"Val Loss: {val_total:.4f} | Time: {elapsed:.1f}s"
    )

# Restore best checkpoint
if best_model_state is not None:
    model.load_state_dict(best_model_state)
print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_range, train_losses, label="Train Total", marker="o")
axes[0].plot(epochs_range, val_losses,   label="Val Total",   marker="s")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Total Loss")
axes[0].legend()

axes[1].plot(epochs_range, train_cls_losses,  label="Cls",  marker="o")
axes[1].plot(epochs_range, train_reg_losses,  label="Reg",  marker="^")
axes[1].plot(epochs_range, train_mask_losses, label="Mask", marker="s")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].set_title("Train Loss Components")
axes[1].legend()

axes[2].plot(epochs_range, val_cls_losses,  label="Val Cls",  marker="o")
axes[2].plot(epochs_range, val_mask_losses, label="Val Mask", marker="s")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss")
axes[2].set_title("Val Loss Components")
axes[2].legend()

plt.suptitle("Training Curves — Instance Segmentation Model", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 5 — Evaluation & Analysis

### 5.1 Mask IoU and Average Precision

We define **Mask IoU** as the Intersection over Union of two binary masks:
$$\text{mIoU}_{\text{mask}} = \frac{|\hat{M} \cap M^*|}{|\hat{M} \cup M^*|}$$

**Mask AP** follows the standard detection AP curve logic but uses mask IoU as the matching criterion instead of box IoU (see 09-01 for box IoU derivation — not re-taught here).

Note: IoU and NMS are derived in 09-01; we apply the same logic here to mask outputs.

### Part 4 — Combined Detection + Segmentation

We combine the detection head (bounding boxes) with the mask head to produce instance-level predictions: each detected object gets both a bounding box and a per-pixel binary mask within that box.

In [ ]:
def mask_iou(
    pred_mask: torch.Tensor,
    gt_mask: torch.Tensor,
    threshold: float = 0.5,
) -> float:
    """Compute binary mask IoU between a predicted and ground-truth mask.

    The predicted mask is first binarised using the given threshold.

    Args:
        pred_mask: Float tensor of shape (H, W) with predicted mask probabilities.
        gt_mask: Binary float tensor of shape (H, W) — ground-truth mask.
        threshold: Probability threshold for binarising pred_mask.

    Returns:
        Scalar float IoU in [0, 1].
    """
    pred_binary = (pred_mask >= threshold).float()
    intersection = (pred_binary * gt_mask).sum().item()
    union = (pred_binary + gt_mask).clamp(max=1).sum().item()
    if union < 1e-6:
        return 1.0 if intersection < 1e-6 else 0.0
    return intersection / union


def compute_precision_recall(
    pred_scores: list[float],
    matched: list[bool],
    num_gt: int,
) -> tuple[list[float], list[float]]:
    """Compute precision-recall curve from sorted detections.

    Predictions are sorted by descending score and accumulated to build the
    precision-recall curve.

    Args:
        pred_scores: Confidence score for each prediction.
        matched: Boolean list indicating whether each prediction is a true positive.
        num_gt: Total number of ground-truth instances.

    Returns:
        Tuple (precision_list, recall_list) of equal length.
    """
    sorted_idx = sorted(range(len(pred_scores)), key=lambda i: pred_scores[i], reverse=True)
    tp_cumulative = 0
    fp_cumulative = 0
    precisions: list[float] = []
    recalls:    list[float] = []

    for idx in sorted_idx:
        if matched[idx]:
            tp_cumulative += 1
        else:
            fp_cumulative += 1
        precision = tp_cumulative / (tp_cumulative + fp_cumulative)
        recall    = tp_cumulative / max(num_gt, 1)
        precisions.append(precision)
        recalls.append(recall)

    return precisions, recalls


def average_precision_mask(
    pred_masks: list[torch.Tensor],
    pred_scores: list[float],
    gt_masks: list[torch.Tensor],
    iou_thresh: float = 0.5,
) -> float:
    """Compute mask Average Precision at a given IoU threshold.

    Sorts predictions by descending score and greedily matches each prediction
    to an unmatched GT mask with mask IoU above the threshold. Computes AP
    using the trapezoid rule on the precision-recall curve.

    Args:
        pred_masks: List of predicted binary mask tensors (H, W).
        pred_scores: Corresponding confidence scores.
        gt_masks: List of ground-truth binary mask tensors (H, W).
        iou_thresh: IoU threshold for a TP match. Defaults to 0.5.

    Returns:
        Average Precision scalar in [0, 1].
    """
    if len(gt_masks) == 0:
        return 0.0
    if len(pred_masks) == 0:
        return 0.0

    num_gt = len(gt_masks)
    gt_matched = [False] * num_gt
    matched_flags: list[bool] = []

    # Sort predictions by descending score
    sorted_idx = sorted(range(len(pred_scores)), key=lambda i: pred_scores[i], reverse=True)

    for pred_idx in sorted_idx:
        pred_m = pred_masks[pred_idx]
        best_iou = 0.0
        best_gt_idx = -1

        for gt_idx, gt_m in enumerate(gt_masks):
            if gt_matched[gt_idx]:
                continue
            iou_val = mask_iou(pred_m, gt_m)
            if iou_val > best_iou:
                best_iou    = iou_val
                best_gt_idx = gt_idx

        if best_iou >= iou_thresh and best_gt_idx >= 0:
            matched_flags.append(True)
            gt_matched[best_gt_idx] = True
        else:
            matched_flags.append(False)

    # Build precision-recall curve
    precisions, recalls = compute_precision_recall(
        pred_scores=[pred_scores[i] for i in sorted_idx],
        matched=matched_flags,
        num_gt=num_gt,
    )

    # Area under PR curve (trapezoid)
    if len(precisions) < 2:
        return precisions[0] if precisions else 0.0

    ap = 0.0
    for k in range(1, len(recalls)):
        ap += (recalls[k] - recalls[k - 1]) * precisions[k]
    return max(0.0, ap)


# ── Per-instance mask IoU on test samples ─────────────────────────────────
model.eval()
mask_iou_scores: list[float] = []
box_iou_scores:  list[float] = []

def box_iou_score(box_a: torch.Tensor, box_b: torch.Tensor) -> float:
    """Compute IoU between two axis-aligned boxes. See 09-01 for full derivation."""
    inter_x1 = max(box_a[0].item(), box_b[0].item())
    inter_y1 = max(box_a[1].item(), box_b[1].item())
    inter_x2 = min(box_a[2].item(), box_b[2].item())
    inter_y2 = min(box_a[3].item(), box_b[3].item())
    inter_w  = max(0.0, inter_x2 - inter_x1)
    inter_h  = max(0.0, inter_y2 - inter_y1)
    inter    = inter_w * inter_h
    area_a   = (box_a[2] - box_a[0]).item() * (box_a[3] - box_a[1]).item()
    area_b   = (box_b[2] - box_b[0]).item() * (box_b[3] - box_b[1]).item()
    union    = area_a + area_b - inter
    return inter / (union + 1e-6)


with torch.no_grad():
    for img, targets in test_data[:20]:
        if len(targets) == 0:
            continue
        img_batch   = img.unsqueeze(0).to(device)
        boxes_batch = [torch.stack([t["box"] for t in targets]).to(device)]

        cls_logits, box_preds, mask_logits = model(img_batch, boxes_batch)
        pred_probs = torch.sigmoid(mask_logits)  # (N, num_classes, 14, 14)

        for inst_idx, tgt in enumerate(targets):
            cls_i   = tgt["class"]
            gt_mask = tgt["mask"]   # (128, 128)
            gt_box  = tgt["box"]

            # Upscale predicted mask to image size for fair IoU comparison
            pred_mask_small = pred_probs[inst_idx, cls_i]  # (14, 14)
            pred_mask_full  = F.interpolate(
                pred_mask_small.unsqueeze(0).unsqueeze(0),
                size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
            ).squeeze()  # (128, 128)

            mask_iou_val = mask_iou(pred_mask_full.cpu(), gt_mask)
            mask_iou_scores.append(mask_iou_val)

            # Box IoU: we use GT box as both proposal and reference → IoU=1 (oracle mode)
            # Predicted delta is added to compute "predicted" box — here delta ≈ 0
            predicted_box = gt_box + box_preds[inst_idx].cpu()
            box_iou_val   = box_iou_score(predicted_box, gt_box)
            box_iou_scores.append(box_iou_val)

print(f"Evaluated {len(mask_iou_scores)} instances from 20 test images")
print(f"Mean mask IoU : {np.mean(mask_iou_scores):.4f}")
print(f"Median mask IoU: {np.median(mask_iou_scores):.4f}")
print(f"Mean box IoU  : {np.mean(box_iou_scores):.4f}")

In [ ]:
# ── Mask AP on test set ────────────────────────────────────────────────────
all_pred_masks:  list[torch.Tensor] = []
all_pred_scores: list[float]        = []
all_gt_masks:    list[torch.Tensor] = []

model.eval()
with torch.no_grad():
    for img, targets in test_data[:50]:
        if len(targets) == 0:
            continue
        img_batch   = img.unsqueeze(0).to(device)
        boxes_batch = [torch.stack([t["box"] for t in targets]).to(device)]

        cls_logits, _, mask_logits = model(img_batch, boxes_batch)
        pred_probs     = torch.sigmoid(mask_logits)  # (N, C, 14, 14)
        cls_probs      = torch.softmax(cls_logits, dim=-1)  # (N, C+1)

        for inst_idx, tgt in enumerate(targets):
            cls_i   = tgt["class"]
            gt_mask = tgt["mask"]

            pred_mask_small = pred_probs[inst_idx, cls_i]  # (14, 14)
            pred_mask_full  = F.interpolate(
                pred_mask_small.unsqueeze(0).unsqueeze(0),
                size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
            ).squeeze().cpu()

            score = cls_probs[inst_idx, cls_i].item()
            all_pred_masks.append(pred_mask_full)
            all_pred_scores.append(score)
            all_gt_masks.append(gt_mask)

ap_50  = average_precision_mask(all_pred_masks, all_pred_scores, all_gt_masks, iou_thresh=0.50)
ap_75  = average_precision_mask(all_pred_masks, all_pred_scores, all_gt_masks, iou_thresh=0.75)

print(f"Mask AP@0.50 (50 test images): {ap_50:.4f}")
print(f"Mask AP@0.75 (50 test images): {ap_75:.4f}")

results_df = pd.DataFrame({
    "Metric": ["Mask AP@0.50", "Mask AP@0.75", "Mean Mask IoU"],
    "Score":  [ap_50, ap_75, np.mean(mask_iou_scores)],
})
results_df.style.highlight_max(subset=["Score"])

In [ ]:
# ── Analysis 1: RoI Pool vs RoI Align comparison on trained backbone ───────
# We compare alignment quality using the trained model's feature map
model.eval()
sample_img, sample_targets = test_data[0]
with torch.no_grad():
    sample_feat = model.backbone(sample_img.unsqueeze(0).to(device))[0]  # (64, 16, 16)

sample_box_img   = sample_targets[0]["box"]  # (4,) image coords
sample_box_feat  = sample_box_img / 8.0      # scale to feature coords
sample_box_feat  = sample_box_feat.unsqueeze(0)  # (1, 4)

pool_result  = roi_pool(sample_feat.cpu(), sample_box_feat, output_size=ROI_OUTPUT_SIZE)
align_result = roi_align(sample_feat.cpu(), sample_box_feat, output_size=ROI_OUTPUT_SIZE)

pool_l2  = pool_result.pow(2).mean().sqrt().item()
align_l2 = align_result.pow(2).mean().sqrt().item()

print("RoI Pooling vs RoI Align on trained backbone feature map:")
print(f"  Pool  RMS response : {pool_l2:.4f}")
print(f"  Align RMS response : {align_l2:.4f}")

# Show single-channel view of the result
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(pool_result[0, 0].numpy(), cmap="viridis")
axes[0].set_title(f"RoI Pool output (ch 0)\n{ROI_OUTPUT_SIZE}×{ROI_OUTPUT_SIZE}")
axes[0].axis("off")
axes[1].imshow(align_result[0, 0].numpy(), cmap="viridis")
axes[1].set_title(f"RoI Align output (ch 0)\n{ROI_OUTPUT_SIZE}×{ROI_OUTPUT_SIZE}")
axes[1].axis("off")
plt.suptitle("RoI Pool vs RoI Align — Trained Backbone Feature Map", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Analysis 2: Box IoU vs Mask IoU scatter ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(box_iou_scores, mask_iou_scores, alpha=0.7, edgecolors="k", linewidths=0.5, s=60)
ax.set_xlabel("Box IoU (predicted vs GT box)", fontsize=11)
ax.set_ylabel("Mask IoU", fontsize=11)
ax.set_title("Box IoU vs Mask IoU — Test Instances", fontsize=12)
ax.axhline(y=0.5, color="red",   linestyle="--", alpha=0.6, label="Mask IoU = 0.50")
ax.axvline(x=0.5, color="green", linestyle="--", alpha=0.6, label="Box IoU = 0.50")
ax.legend(fontsize=9)
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

correlation = np.corrcoef(box_iou_scores, mask_iou_scores)[0, 1]
print(f"Pearson correlation (box IoU vs mask IoU): {correlation:.4f}")

In [ ]:
# ── Analysis 3: Qualitative predictions on test set ────────────────────────
model.eval()
fig, axes = plt.subplots(3, 4, figsize=(14, 10))

with torch.no_grad():
    for row_idx in range(3):
        img, targets = test_data[row_idx]
        if len(targets) == 0:
            continue
        img_batch   = img.unsqueeze(0).to(device)
        boxes_batch = [torch.stack([t["box"] for t in targets]).to(device)]

        cls_logits, box_preds, mask_logits = model(img_batch, boxes_batch)
        pred_probs = torch.sigmoid(mask_logits).cpu()  # (N, C, 14, 14)
        cls_preds  = cls_logits.argmax(dim=-1).cpu()   # (N,)

        img_np = img.permute(1, 2, 0).numpy()

        # Col 0: original image + GT boxes
        ax = axes[row_idx, 0]
        ax.imshow(img_np)
        for tgt in targets:
            x1, y1, x2, y2 = tgt["box"].numpy()
            c = tgt["class"]
            rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=2,
                                       edgecolor=["red","green","blue"][c],
                                       facecolor="none")
            ax.add_patch(rect)
        ax.set_title(f"Image {row_idx}\nGT boxes ({len(targets)} inst.)", fontsize=8)
        ax.axis("off")

        # Col 1: GT masks overlay
        ax = axes[row_idx, 1]
        ax.imshow(img_np)
        for tgt in targets:
            mask_np = tgt["mask"].numpy()
            cls_c   = tgt["class"]
            rgba    = np.zeros((*mask_np.shape, 4))
            colours = [[1,0,0,0.5], [0,1,0,0.5], [0,0,1,0.5]]
            for ch_i in range(4):
                rgba[:,:,ch_i] = mask_np * colours[cls_c][ch_i]
            ax.imshow(rgba)
        ax.set_title("GT masks", fontsize=8)
        ax.axis("off")

        # Col 2: predicted masks overlay
        ax = axes[row_idx, 2]
        ax.imshow(img_np)
        for inst_i, tgt in enumerate(targets):
            cls_gt = tgt["class"]
            pred_small = pred_probs[inst_i, cls_gt]  # (14, 14)
            pred_full  = F.interpolate(
                pred_small.unsqueeze(0).unsqueeze(0),
                size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
            ).squeeze().numpy()
            rgba = np.zeros((*pred_full.shape, 4))
            colours = [[1,0,0,0.5], [0,1,0,0.5], [0,0,1,0.5]]
            pred_binary = (pred_full > 0.5).astype(float)
            for ch_i in range(4):
                rgba[:,:,ch_i] = pred_binary * colours[cls_gt][ch_i]
            ax.imshow(rgba)
        ax.set_title("Predicted masks", fontsize=8)
        ax.axis("off")

        # Col 3: mask IoU per instance
        ax = axes[row_idx, 3]
        iou_vals = []
        for inst_i, tgt in enumerate(targets):
            cls_gt    = tgt["class"]
            pred_small = pred_probs[inst_i, cls_gt]
            pred_full  = F.interpolate(
                pred_small.unsqueeze(0).unsqueeze(0),
                size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
            ).squeeze()
            iou_v = mask_iou(pred_full, tgt["mask"])
            iou_vals.append(iou_v)
        ax.bar(range(len(iou_vals)), iou_vals,
               color=[["red","green","blue"][tgt["class"]] for tgt in targets],
               alpha=0.75)
        ax.axhline(0.5, linestyle="--", color="black", alpha=0.5)
        ax.set_xlabel("Instance"); ax.set_ylabel("Mask IoU")
        ax.set_title(f"Mask IoU\nMean={np.mean(iou_vals):.3f}", fontsize=8)
        ax.set_ylim(0, 1.05)

plt.suptitle("Test Set Predictions — GT vs Predicted Masks", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Failure mode analysis: find instances with lowest mask IoU ─────────────
# Collect IoU + sample info for all test instances
instance_records: list[dict] = []
model.eval()

with torch.no_grad():
    for img_idx, (img, targets) in enumerate(test_data):
        if len(targets) == 0:
            continue
        img_batch   = img.unsqueeze(0).to(device)
        boxes_batch = [torch.stack([t["box"] for t in targets]).to(device)]
        _, _, mask_logits = model(img_batch, boxes_batch)
        pred_probs = torch.sigmoid(mask_logits).cpu()

        for inst_i, tgt in enumerate(targets):
            cls_gt     = tgt["class"]
            pred_small = pred_probs[inst_i, cls_gt]
            pred_full  = F.interpolate(
                pred_small.unsqueeze(0).unsqueeze(0),
                size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
            ).squeeze()
            iou_val = mask_iou(pred_full, tgt["mask"])
            instance_records.append({
                "img_idx":  img_idx,
                "inst_idx": inst_i,
                "cls":      cls_gt,
                "iou":      iou_val,
                "img":      img,
                "tgt":      tgt,
                "pred":     pred_full,
            })

# Sort by mask IoU ascending — worst cases first
instance_records.sort(key=lambda r: r["iou"])
worst_3 = instance_records[:3]

fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for failure_row, record in enumerate(worst_3):
    img_np    = record["img"].permute(1, 2, 0).numpy()
    gt_mask   = record["tgt"]["mask"].numpy()
    pred_mask = record["pred"].numpy()
    iou_val   = record["iou"]
    cls_i     = record["cls"]
    box_arr   = record["tgt"]["box"].numpy()

    # Image + GT box
    ax = axes[failure_row, 0]
    ax.imshow(img_np)
    x1, y1, x2, y2 = box_arr
    rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                edgecolor=["red","green","blue"][cls_i], facecolor="none")
    ax.add_patch(rect)
    ax.set_title(f"Failure {failure_row+1}\nMask IoU={iou_val:.3f}", fontsize=9)
    ax.axis("off")

    # GT mask
    ax = axes[failure_row, 1]
    ax.imshow(gt_mask, cmap="gray", vmin=0, vmax=1)
    ax.set_title("GT Mask", fontsize=9)
    ax.axis("off")

    # Predicted mask
    ax = axes[failure_row, 2]
    ax.imshow(pred_mask, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Pred Mask (thresholded at 0.5)\nIoU={iou_val:.3f}", fontsize=9)
    ax.axis("off")

plt.suptitle("Failure Mode Analysis — 3 Instances with Lowest Mask IoU", fontsize=12)
plt.tight_layout()
plt.show()

print("Failure mode observations:")
print("  - Very small instances → low RoI feature resolution → blurry masks")
print("  - Instances near image edges → partial feature coverage in RoI Align")
print("  - Low-contrast instances → backbone cannot distinguish foreground well")

---
## Part 6 — Summary & Lessons Learned

### Key Takeaways

1. **Three levels of dense prediction:** Semantic segmentation labels every pixel with a class (09-03); instance segmentation further assigns each pixel to a *specific* object instance; panoptic segmentation unifies both by combining *stuff* (semantic) and *things* (instance) labels.

2. **RoI Pooling introduces quantisation error.** Because bin boundaries are rounded to integer pixel coordinates, features can be misaligned by up to the stride value. On a feature map with stride 8, a single-pixel error translates to 8 image pixels — fatal for pixel-accurate mask prediction.

3. **RoI Align eliminates quantisation** by sampling at exact sub-pixel locations using bilinear interpolation. Even with `sampling_ratio=2` (4 points per bin), the alignment RMSE is measurably lower than RoI Pooling — and this advantage compounds as model depth and stride increase.

4. **The mask loss trains only the GT class channel.** Supervising all class channels simultaneously would cause them to compete. By isolating the GT class prediction, the mask head learns class-specific shape information without interfering cross-class gradients.

5. **Oracle proposals are a useful pedagogical tool.** Using GT boxes as proposals removes the RPN from the learning problem, letting us focus on RoI Align and mask quality in isolation. In production Mask R-CNN, a learned RPN generates noisy proposals with varying IoU overlap, making the mask task substantially harder.

### What's Next

→ **09-05 (Vision Transformer — ViT)** replaces the CNN backbone entirely with a patch-based self-attention architecture, reusing the Transformer encoder blocks from Module 08.

### Going Further

- **Mask R-CNN** (He et al., 2017) — the canonical instance segmentation paper; adds an RPN and uses RoI Align exactly as implemented here.
- **SOLO / SOLOv2** (Wang et al., 2020) — an anchor-free instance segmentation approach that predicts instance masks directly from grid cells without RoI operations.
- **Panoptic FPN** (Kirillov et al., 2019) — extends Mask R-CNN with a semantic segmentation branch to produce panoptic outputs in one forward pass.
- **CondInst / YOLACT** — real-time instance segmentation using mask prototypes and lightweight instance-specific coefficients.